# Thread Pools vs Process Pools in Netrun

This notebook demonstrates how to configure and use both thread and process pools in netrun for parallel execution.

**Key Concepts:**
- Thread pools allow multiple workers to process nodes concurrently
- Process pools run workers in separate processes for true CPU parallelism
- Pools are configured in the `pools` section of NetConfig
- Nodes are assigned to pools via `execution_config.pools`
- Factory-based nodes work with process pools via lazy resolution

**Tip:** You can visualize and edit the network configuration by running `netrun-ui` in this folder.

## Pool Types Overview

| Pool Type | Description | Best For |
|-----------|-------------|----------|
| `main` | Single worker in main event loop | Lightweight async operations |
| `thread` | Multiple worker threads | I/O-bound work, concurrent tasks |
| `multiprocess` | Separate processes | CPU-bound work (requires picklable functions) |
| `remote` | Network-connected workers | Distributed execution |

In [1]:
from nodes import *

In [2]:
find_primes(print, 0, 10)

Finding 10 primes starting from 0
Found 10 primes: 2...29


[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

## Load the Configuration

In [1]:
import json
import time
from copy import deepcopy
from pathlib import Path

from netrun.core import Net, NetConfig

# Load the base configuration
config_path = Path("main.netrun.json")
config_data = json.loads(config_path.read_text())

print("Pool configuration from file:")
print(json.dumps(config_data["pools"], indent=2))

Pool configuration from file:
{
  "main": {
    "spec": {
      "type": "main"
    }
  },
  "thread_pool": {
    "spec": {
      "type": "thread",
      "num_workers": 4
    }
  },
  "process_pool": {
    "spec": {
      "type": "multiprocess",
      "num_workers": 4
    }
  }
}


## Helper Function to Run with Different Pool Types

In [2]:
async def run_with_pool(base_config: dict, pool_type: str, num_workers: int) -> float:
    """Run the network with a specific pool type/worker count and return elapsed time."""
    config = deepcopy(base_config)
    
    # Configure pool with specified type and workers
    config["pools"] = {
        "compute_pool": {
            "spec": {"type": pool_type, "num_workers": num_workers}
        },
        "main": {"spec": {"type": "main"}},
    }
    
    # Update all hash nodes to use compute_pool
    for node in config["graph"]["nodes"]:
        if node["name"].startswith("hash_"):
            node["execution_config"] = {"pools": ["compute_pool"]}
    
    net_config = NetConfig.model_validate(config)
    
    start_time = time.perf_counter()
    
    async with Net(net_config) as net:
        # Inject data for each hash node
        for i, name in enumerate(["alpha", "beta", "gamma", "delta"], 1):
            net.inject_data(f"hash_{i}", "data", [name])
            net.inject_data(f"hash_{i}", "iterations", [200_000])
        
        # Run until complete
        while True:
            await net.run_until_blocked()
            startable = net.get_startable_epochs()
            if not startable:
                break
            for epoch_id in startable:
                await net.execute_epoch(epoch_id)
        
        results = net.get_all_outputs("results")
    
    elapsed = time.perf_counter() - start_time
    return elapsed

## Thread Pool: 1 Worker (Sequential)

In [3]:
print("Running with 1 thread (sequential)...")
time_t1 = await run_with_pool(config_data, "thread", 1)
print(f"Completed in {time_t1:.2f}s")

Running with 1 thread (sequential)...
Completed in 0.29s


## Thread Pool: 4 Workers (Concurrent)

In [4]:
print("Running with 4 threads (concurrent)...")
time_t4 = await run_with_pool(config_data, "thread", 4)
print(f"Completed in {time_t4:.2f}s")

Running with 4 threads (concurrent)...
Completed in 0.29s


## Process Pool: 1 Worker (Sequential)

In [5]:
print("Running with 1 process (sequential)...")
time_p1 = await run_with_pool(config_data, "multiprocess", 1)
print(f"Completed in {time_p1:.2f}s")

Running with 1 process (sequential)...
Completed in 0.41s


## Process Pool: 4 Workers (Parallel)

In [6]:
print("Running with 4 processes (parallel)...")
time_p4 = await run_with_pool(config_data, "multiprocess", 4)
print(f"Completed in {time_p4:.2f}s")

Running with 4 processes (parallel)...
Completed in 0.39s


## Compare Results

In [7]:
print("=" * 50)
print("Performance Comparison")
print("=" * 50)
print(f"Thread pool  - 1 worker:  {time_t1:.2f}s")
print(f"Thread pool  - 4 workers: {time_t4:.2f}s  ({time_t1 / time_t4:.2f}x)")
print(f"Process pool - 1 worker:  {time_p1:.2f}s")
print(f"Process pool - 4 workers: {time_p4:.2f}s  ({time_p1 / time_p4:.2f}x)")
print()
print("Thread pools are limited by Python's GIL for CPU-bound work.")
print("Process pools bypass the GIL and achieve true parallelism.")

Performance Comparison
Thread pool  - 1 worker:  0.29s
Thread pool  - 4 workers: 0.29s  (1.01x)
Process pool - 1 worker:  0.41s
Process pool - 4 workers: 0.39s  (1.05x)

Thread pools are limited by Python's GIL for CPU-bound work.
Process pools bypass the GIL and achieve true parallelism.


## Understanding the GIL

Python's **Global Interpreter Lock (GIL)** prevents true parallel execution of CPU-bound Python code in threads. This means:

- **Thread pools** are best for **I/O-bound** work (network requests, file I/O)
- **Multiprocess pools** are best for **CPU-bound** work (calculations)

However, thread pools still provide benefits for:
- Concurrent I/O operations
- Operations that release the GIL (numpy, etc.)
- Overlapping computation with I/O

## Pool Configuration Reference

Here's how to configure pools in your `main.netrun.json`:

```json
{
  "pools": {
    "main": {
      "spec": {"type": "main"}
    },
    "thread_pool": {
      "spec": {
        "type": "thread",
        "num_workers": 4
      }
    },
    "process_pool": {
      "spec": {
        "type": "multiprocess",
        "num_workers": 4
      }
    }
  },
  "graph": {
    "nodes": [
      {
        "name": "my_node",
        "factory": "netrun.node_factories.from_function",
        "factory_args": {"func": "nodes.my_func"},
        "execution_config": {
          "pools": ["process_pool"]
        }
      }
    ]
  }
}
```

## Understanding the Node Functions

Let's look at the node functions defined in `nodes.py`:

In [8]:
print(Path("nodes.py").read_text())

"""Node functions demonstrating CPU-bound work for pool comparison.

This module contains functions that perform CPU-intensive calculations.
When run in a thread pool, they are limited by Python's GIL (Global Interpreter Lock).
When run in a multiprocess pool, they can utilize multiple CPU cores in parallel.
"""

import hashlib


def compute_hash(data: str, iterations: int, print) -> dict:
    """Compute a hash iteratively (CPU-bound work).

    This simulates CPU-intensive work by repeatedly hashing a value.
    """
    print(f"Starting hash computation with {iterations} iterations")

    result = data.encode()
    for i in range(iterations):
        result = hashlib.sha256(result).digest()
        if (i + 1) % (iterations // 4) == 0:
            print(f"Progress: {(i + 1) * 100 // iterations}%")

    hex_result = result.hex()[:16]
    print(f"Completed: {hex_result}...")

    return {"input": data, "iterations": iterations, "hash": hex_result}


def is_prime(n: int) -> bool:
    """C